# 01 - Study Area Definition and SNOTEL Data Download
This notebook will be used to download some of the various datasets needed for the project. That includes:

| Dataset | Use | Code Complete? | Downloaded? |
| --- | --- | --- | --- |
| USGS/other federal boundary holder | site boundary that's used to select downloaded data | yes | yes |
| USGS DEM | digital elevation model used for SWE training | no | no |
| SNOTEL | timeseries used for SWE training | yes | yes |
| BCQC SNOTEL | bias corrected SNOTEL data | yes | yes |
| PRISM climate data | gridded reanalysis data used for SWE training | no | yes |
| MACAv2 climate data | gridded future data used for SWE prediction | no | no |

## Step 1: Import Libraries and Setup

In [1]:
# File management
import os
import pathlib
from pathlib import Path
import pyarrow # saving GeoParquet files
import sys

# Downloading
from tqdm.notebook import tqdm # progress bar
import requests # for SNOTEL API access
import time
import zipfile

# Data Management
import pandas as pd
import xarray as xr

# Geospatial Management
import geopandas as gpd
from pygeohydro import WBD # site boundary based on watersheds
#import rasterio
import rioxarray as rxr
from shapely.geometry import box # lat/lon boundary boxes
from shapely.ops import split
from shapely.ops import unary_union # for intersecting boundaries

# Plotting
import matplotlib
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas

In [2]:
# Force Jupyter to reload external modules automatically before executing a cell
# I used Gemini to help me set up the connection to my src files

# force reimporting of src scripts
%load_ext autoreload
%autoreload 2

# Find the parent directory of this notebook (the repo root) and add it to Python's search path
repo_root = str(Path.cwd().parent)
if repo_root not in sys.path:
    sys.path.append(repo_root)

# Now Python can see the src folder
# import just specific functions
from src import snotel

# once I'm using environment.yml I can get rid of the above steps

# from src
from src import snotel

print('src import complete!')

src import complete!


In [5]:
# You may need to install pygeohydro 

# if so, uncomment and run this block to install the 
# package to your current kernel

#%pip install pygeohydro

# For windows users - also run this

#%pip uninstall -y aiodns

In [3]:
# set directories
proj_dir = os.path.join(pathlib.Path.home(),
                        'Documents',
                        'Graduate_School',
                        'EDA_Certificate', 
                        'Summer', 
                        'snow-drought-modeling')
os.makedirs(proj_dir, exist_ok=True)

raw_data_dir = os.path.join(proj_dir, 'data', 'raw')
os.makedirs(raw_data_dir, exist_ok=True)

cleaned_data_dir = os.path.join(proj_dir, 'data', 'cleaned')
os.makedirs(raw_data_dir, exist_ok=True)

## Step 2: Site Selection
In this step, I download the HUC6 watershed boundary for the Missouri Headwaters region to use as the bounding box for the project.

I've selected HUC6 watershed boundaries that will capture the following ranges:

**FILL IN**

**REASONING FOR HUC6 boundaries**

documentation on pygeohydro: https://docs.hyriver.io/readme/pygeohydro.html

Citation:
@article{Chegini_2021,
    author = {Chegini, Taher and Li, Hong-Yi and Leung, L. Ruby},
    doi = {10.21105/joss.03175},
    journal = {Journal of Open Source Software},
    month = {10},
    number = {66},
    pages = {1--3},
    title = {{HyRiver: Hydroclimate Data Retriever}},
    volume = {6},
    year = {2021}
}

### 2a: Download

In [4]:
# get HUC8 boundaries in SW Montana bounding box

# set bounding box
bbox = (-114.5, 44.3, -109.5, 47.0) # this is bigger than the ultimate study area

# get all HUC8 boundaries w/ WBD package
wbd = WBD("huc8")

# filter to bounding box
huc8_bbox = wbd.bygeom(box(*bbox), geo_crs=4326)

# Check total HUC8s in box
print(f"Total huc8s in bounding box: {len(huc8_bbox)}")
print(huc8_bbox[["huc8", "name"]].to_string())


Total huc8s in bounding box: 44
        huc8                          name
0   10030102       Upper Missouri-Dearborn
1   10070003                       Shields
2   10020006                       Boulder
3   10030103                         Smith
4   10030105                          Belt
5   10040103                        Judith
6   10040201             Upper Musselshell
7   17040202                  Upper Henrys
8   17040214                  Beaver-Camas
9   17040215                Medicine Lodge
10  17040216                         Birch
11  17040217                   Little Lost
12  17060201                  Upper Salmon
13  17060202                    Pahsimeroi
14  17060203         Middle Salmon-Panther
15  17060204                         Lemhi
16  17060205      Upper Middle Fork Salmon
17  17060206      Lower Middle Fork Salmon
18  17060207     Middle Salmon-Chamberlain
19  17060301                  Upper Selway
20  17060302                  Lower Selway
21  17060303          

In [5]:
# Filter to Missouri Headwaters headwaters (huc6 = 100200)
huc8_missouri = huc8_bbox[huc8_bbox["huc8"].str.startswith("1002")].copy()

# Check HUC8s in Missouri Headwaters
print(f"\nMissouri Headwaters HUC8s: {len(huc8_missouri)}")
print(huc8_missouri[["huc8", "name"]].to_string())



Missouri Headwaters HUC8s: 8
        huc8        name
2   10020006     Boulder
29  10020001    Red Rock
30  10020002  Beaverhead
31  10020003        Ruby
32  10020004    Big Hole
33  10020005   Jefferson
38  10020008    Gallatin
40  10020007     Madison


In [6]:
# Visual Inspection

# Set CRS
huc8_missouri = huc8_missouri.set_crs("EPSG:4326")

# Make plot
huc8_plot = huc8_missouri.hvplot.polygons(
    geo=True,
    tiles="EsriNatGeo",
    alpha=0.4,
    line_color="black",
    line_width=0.8,
    color="name",
    cmap="tab20",
    hover_cols=["huc8", "name"],
    title="Missouri Headwaters HUC8 Watersheds — SW Montana",
    width=900,
    height=700,
    legend=False
)

# Show plot
huc8_plot

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]   (name,huc8)

This is currently missing SNOTEL sites in the Bridger range, and potentially in the Gallatins. Further, there might be some SNOTEL sites in the western Absarokas that would be worth including in the training dataset. Worth looking into adding more HUC8 areas. The western and southern boundaries look good (that's the continental divide); the northern boundary could catch the rest of the mountains to Helena but that might be too far of a reach.

https://ftpgeoinfo.msl.mt.gov/Data/Spatial/NonMSDI/NRCS/HUCEnv/MTHydroMap.pdf great visualization of all the HUC8s. Maybe add Shields, Upper Yellowstone, Yellowstone Headwaters

In [6]:
print(huc8_bbox[["huc8", "name"]].to_string())

        huc8                          name
0   10030102       Upper Missouri-Dearborn
1   10070003                       Shields
2   10020006                       Boulder
3   10030103                         Smith
4   10030105                          Belt
5   10040103                        Judith
6   10040201             Upper Musselshell
7   17040202                  Upper Henrys
8   17040214                  Beaver-Camas
9   17040215                Medicine Lodge
10  17040216                         Birch
11  17040217                   Little Lost
12  17060201                  Upper Salmon
13  17060202                    Pahsimeroi
14  17060203         Middle Salmon-Panther
15  17060204                         Lemhi
16  17060205      Upper Middle Fork Salmon
17  17060206      Lower Middle Fork Salmon
18  17060207     Middle Salmon-Chamberlain
19  17060301                  Upper Selway
20  17060302                  Lower Selway
21  17060303                        Lochsa
22  1701020

I'll do a basic search on the [SNOTEL API Demo](https://wcc.sc.egov.usda.gov/awdbRestApi/swagger-ui/index.html#/Station%20Metadata/getStations) to see how many SNOTEL sites are in the three watersheds I'll consider adding:
- Shields (10070003): 4 SNOTEL sites (2 in Bridgers, 2 in Crazy Mountains)
- Upper Yellowstone (10070002): 5 SNOTEL sites, goes fairly far east
- Yellowstone Headwaters (10070001): 1 SNOTEL site in MT, 4 in WY

So, might be some important sites to capture here, or it could really spread out the study area. I might need to plot these SNOTEL sites on a map to see where they land.

Using the [MT State Library ArcGIS viewer](https://www.arcgis.com/apps/mapviewer/index.html?layers=1b0d0b21de9747d0b6a16e3428ae1eb6), I've found a few SNOTEL sites that are within 3 miles of the Missouri Headwaters watershed boundary:
- Brackett Creek (Bridgers)
- Sacajawea (Bridgers)
- Canyon (Yellowstone) 384:WY:SNTL
- White Elephant (Henry's Fork)
- Saddle Mtn (Lower East Fork Bitterroot)
- Moose Creek (Salmon)
- Barker Lakes (Upper Clark Fork)
- Frohner Meadow (Upper Missouri)

Based on these extra sites, I can finalize study area design. A 5km buffer around the Missouri Headwaters Basin will grab a few extra SNOTEL sites that seem important for the training data. However, a few lie on the other side of the Continental Divide, so I'll clip those out after filtering the sites. I can proceed with the study area boundary generation below using a spatial join and don't need to clumsily tack on extra HUCs as I thought. Then, in the SNOTEL processing step, I will grab metadata for all MT sites and filter to the study area. The table below - inspired by a conversation with Claude - cleanly explains the boundaries I'm using:

| Layer | Definition | Purpose |
| --- | --- | --- |
| Study area | HUC6 100200 (Missouri Headwaters Basin) dissolved | Spatial study definition, result maps |
| Buffer area | Study area + 5km buffer | SNOTEL, PRISM, MACA data selection |
| Continental Divide Clip | Buffer area clipped to remove buffer on the west side of the CD | SNOTEL, PRISM, MACA data selection |

### 2b: Spatial Join
need to join and dissolve

In [7]:
# Set study area boundary
mhw_gdf = huc8_missouri.dissolve().reset_index()[['geometry']]
mhw_gdf = mhw_gdf.set_crs('EPSG:4326')

# check it out
mhw_gdf

,geometry
0,"POLYGON ((-111.51646 44.64282, -111.51652 44.6..."


### 2c: Plot Boundary

In [8]:
# make the plot
study_area_plot = mhw_gdf.hvplot(
    geo=True,
    tiles="EsriNatGeo",
    alpha=0.4,
    line_color="black",
    line_width=0.8,
    title="Missouri Headwaters Basin (HUC6) Study Area",
    width=900,
    height=700,
    legend=False
)

# Show the plot
study_area_plot

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

### 2d: Buffer study area and clip to east side of Continental Divide

From the visual inspection I performed above, I found that just using the Missouri Headwaters Basin watershed boundary might exclude some SNOTEL sites that would be appropriate to include. I determine that sites that are appropriate must:
- Lie within the study area
- Or lie within a 5km buffer of the study are
- AND be located east of the continental divide

Based on some visual inspection, it was clear that there were 8 sites that were extremely close to the study area, and would likely be worthwhile to include in the training data. For example, the only two SNOTEL sites in the Bridger Range (Brackett Creek and Sacajawea) are just outside the study area - leaving them out would lose the only ground-truthed record of dynamics for the whole Bridger Range. Thus, I will use a buffered boundary to increase the size of the training dataset. However, I don't want to introduce SNOTEL sites that are potentially impacted by different climate regimes than the study area. Ranges such as the Bridgers are likely driven by the same climate phenomena as the rest of the study area. However, the western edge of the study area is defined by the Continental Divide, and some SNOTEL sites that are just outside the boundary are on the other side of the Divide. This means that these sites might have significantly different climate regimes than the rest of the study area, introducing error. Because of this effect, I will exclude sites west of the Continental Divide.

The workflow will look as follows:
- Create buffered study boundary
- Remove buffer west of Continental Divide
- Select sites within boundary

In [9]:
# Step 1: Buffer study area by ~5km (code derived from Claude conversation)

# Set CRS to UTMs
study_area_bound = mhw_gdf.to_crs("EPSG:32612")

# Set buffer
buffer_proj = study_area_bound.buffer(5000) #5000 meters bc in UTM now

# Convert back to EPSG 4326
mhw_buff_gdf = gpd.GeoDataFrame(
    geometry=buffer_proj
).set_crs("EPSG:32612").to_crs("EPSG:4326")

In [10]:
# Visual inspection

# make a buffer plot
buffer_plot = mhw_buff_gdf.hvplot(
    geo=True,
    fill_color = None,
    line_color="red",
    line_width=0.8,
    width=900,
    height=700,
    legend=False
)

# overlay on study plot
study_buffer_plot = (
    study_area_plot
    *
    buffer_plot 
).opts(title="Missouri Headwaters Basin (HUC6) Study Area with 5km buffer")

# show plot
study_buffer_plot

:Overlay
   .WMTS.I      :WMTS   [Longitude,Latitude]
   .Polygons.I  :Polygons   [Longitude,Latitude]
   .Polygons.II :Polygons   [Longitude,Latitude]

In [11]:
# Step 2: Download Continental Divide shapefile
# Read Continental Divide directly from FWS ArcGIS REST endpoint

divide_url = ("https://services4.arcgis.com/jzlMsmscSx37zEbh/arcgis/rest/"
              "services/Archive_6/FeatureServer/0/query"
              "?where=1%3D1&outFields=*&outSR=4326&f=geojson"
)

continental_divide = gpd.read_file(divide_url)
continental_divide = continental_divide.set_crs("EPSG:4326")

# Clip to region of interest to keep it lightweight
region_box = gpd.GeoDataFrame(
    geometry=[box(-114.5, 44.3, -109.5, 47.0)],
    crs="EPSG:4326"
)
divide_clipped = gpd.clip(continental_divide, region_box) 

In [12]:
# Visual inspection
divide_clipped.hvplot(
    geo=True,
    tiles="EsriNatGeo",
    line_color="black",
    line_width=4,
    title="Continental Divide in Study Region",
    width=900,
    height=700,
    )

:Overlay
   .WMTS.I :WMTS   [Longitude,Latitude]
   .Path.I :Path   [Longitude,Latitude]

This looks right - the boundary I downloaded is a bit long, but that's fine. Proceeding to the final boundary.

In [13]:
# Split the buffer using the Divide line (assisted by Gemini)

# Union the geometries into single Shapely objects to ensure a clean cut
divide_line = divide_clipped.union_all()
buffer_poly = mhw_buff_gdf.union_all()

# Slice the buffered polygon using the line
# This returns a GeometryCollection of the resulting polygon pieces (East and West)
split_collection = split(buffer_poly, divide_line)

# Convert those pieces back into a GeoDataFrame
split_gdf = gpd.GeoDataFrame(
    geometry=list(split_collection.geoms), 
    crs="EPSG:4326"
)

# Visualize
split_gdf.hvplot(geo=True,
    tiles="EsriNatGeo",
    alpha=0.3,
    line_color="black",
    line_width=2,
    width=900,
    height=700)

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

In [14]:

# Keep only the correct side (East of the divide)

# We know the original, unbuffered basin (mhw_gdf) sits safely on the correct side.
# We can find the center of the original basin, and keep whichever buffer piece 
# contains that center point.
original_centroid = mhw_gdf.union_all().centroid

# Filter to the polygon that contains the centroid
mhw_buff_clip_gdf = split_gdf[split_gdf.geometry.contains(original_centroid)].copy()

print(f"Original buffer area split into {len(split_gdf)} pieces.")
print("Successfully isolated the basin-side buffer!")

# visual inspection
buff_clip_plot = mhw_buff_clip_gdf.hvplot(
    geo=True,
    tiles="EsriNatGeo",
    alpha=0.3,
    line_color="red",
    fill_color=None,
    line_width=2,
    width=900,
    height=700
)

# Overlay with study area
mhw_buff_clip_plot = (
    study_area_plot * buff_clip_plot
    ).opts(title="Final Training Area Boundary")

mhw_buff_clip_plot

Original buffer area split into 2 pieces.
Successfully isolated the basin-side buffer!


:Overlay
   .WMTS.I      :WMTS   [Longitude,Latitude]
   .Polygons.I  :Polygons   [Longitude,Latitude]
   .WMTS.II     :WMTS   [Longitude,Latitude]
   .Polygons.II :Polygons   [Longitude,Latitude]

### Step 2e: Save boundaries

In [15]:
# make boundary directories

bound_raw_dir = os.path.join(raw_data_dir, 'boundaries')
bound_final_dir = os.path.join(cleaned_data_dir, 'boundaries')
bound_dirs = [bound_raw_dir, bound_final_dir]
for path in bound_dirs:
    os.makedirs(path, exist_ok=True)

In [16]:
# save boundaries

# raw
mhw_gdf.to_parquet(f'{bound_raw_dir}/mhw_gdf.parquet')
mhw_buff_gdf.to_parquet(f'{bound_raw_dir}/mhw_buff_gdf.parquet')
continental_divide.to_parquet(f'{bound_raw_dir}/cont_div.parquet')

# cleaned
mhw_buff_clip_gdf.to_parquet(f'{bound_final_dir}/mhw_buff_clip_gdf.parquet')

## Step 3: Access SNOTEL Data

- API Documentation: https://wcc.sc.egov.usda.gov/awdbRestApi/v3/api-docs
- Interactive API demo: https://wcc.sc.egov.usda.gov/awdbRestApi/swagger-ui/index.html
- GitHub Demo repo: https://github.com/nrcs-nwcc/iow_awdb_rest_api_demo

### 3a: Download station metadata

This step gets the metadata for each station, which I can then filter using the watershed boundary.

In [19]:
# download station data
variables = ["PREC", "WTEQ", "TMIN", "TMAX"]

mt_stations = snotel.get_snotel_metadata('MT', variables)
wy_stations = snotel.get_snotel_metadata('WY', variables)
id_stations = snotel.get_snotel_metadata('ID', variables)

# Concatenate lists
stations = mt_stations + wy_stations + id_stations
print(len(stations))
print(type(stations))

270
<class 'list'>


[The USDA](https://www.nrcs.usda.gov/state-offices/montana/montana-snow-survey/frequently-asked-snow-survey-questions-montana) confirms the number of stations. Success! There is one station in Wyoming that needs to be added though - I'll download that station and append it.

Note: The sites lying close to the boundary were identified by hand, and I determined that one site I wanted to include was in Wyoming. Other sites that are within the proximity I decided on (5km) are not included in the SNOTEL download, because they lie in Idaho (separated from MT by the Continental Divide), and I would have filtered them out anyways. 

**Potentially revist to pull all ID and WY sites, then filter**

In [20]:
# Convert list to gdf
# make it a df first
station_df = pd.DataFrame(stations)
# then convert to GDF
sntl_station_gdf = gpd.GeoDataFrame(
    station_df,
    geometry=gpd.points_from_xy(station_df['longitude'], station_df['latitude']),
    crs = 'EPSG:4326'
    )

sntl_station_gdf

,stationTriplet,stationId,stateCode,networkCode,name,dcoCode,countyName,huc,elevation,latitude,longitude,dataTimeZone,shefId,operator,beginDate,endDate,associatedHucs,pedonCode,geometry
0,916:MT:SNTL,916,MT,SNTL,Albro Lake,MT,Madison,100200050701,8500.0,45.59723,-111.95902,-8.0,ABRM8,NRCS,1996-09-01 00:00,2100-01-01 00:00,"[100200050601, 100200050702, 100200071101]",NaN,POINT (-111.95902 45.59723)
1,307:MT:SNTL,307,MT,SNTL,Badger Pass,MT,Pondera,100302010201,6870.0,48.13091,-113.02311,-8.0,BADM8,NRCS,1968-10-01 00:00,2100-01-01 00:00,"[100302010202, 100302010204, 100302010601, 100...",NaN,POINT (-113.02311 48.13091)
2,311:MT:SNTL,311,MT,SNTL,Banfield Mountain,MT,Lincoln,170101011202,5580.0,48.57120,-115.44573,-8.0,BANM8,NRCS,1968-10-01 00:00,2100-01-01 00:00,"[170101011001, 170101011002, 170101011106, 170...",NaN,POINT (-115.44573 48.5712)
3,313:MT:SNTL,313,MT,SNTL,Barker Lakes,MT,Deer Lodge,170102010304,8250.0,46.09713,-113.13038,-8.0,BRLM8,NRCS,1977-08-01 00:00,2100-01-01 00:00,"[100200040703, 170102010208, 170102010301]",NaN,POINT (-113.13038 46.09713)
4,315:MT:SNTL,315,MT,SNTL,Basin Creek,MT,Silver Bow,170102010201,7120.0,45.79737,-112.52047,-8.0,BSCM8,NRCS,1976-10-01 00:00,2100-01-01 00:00,"[100200041201, 100200041203, 100200050501, 170...",NaN,POINT (-112.52047 45.79737)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
265,845:ID:SNTL,845,ID,SNTL,Vienna Mine,ID,Blaine,170602010201,8930.0,43.79942,-114.85273,-8.0,VNNI1,NRCS,1978-10-01 00:00,2100-01-01 00:00,"[170501130101, 170501130102, 170501130104, 170...",NaN,POINT (-114.85273 43.79942)
266,855:ID:SNTL,855,ID,SNTL,West Branch,ID,Adams,170501240201,5590.0,45.07220,-116.45413,-8.0,WBRI1,NRCS,1979-10-01 00:00,2100-01-01 00:00,"[170501240101, 170502010501, 170602100102, 170...",NaN,POINT (-116.45413 45.0722)
267,860:ID:SNTL,860,ID,SNTL,White Elephant,ID,Fremont,170402020305,7670.0,44.53267,-111.41085,-8.0,WHEI1,NRCS,1979-10-01 00:00,2100-01-01 00:00,"[100200010101, 170402020102, 170402020103, 170...",NaN,POINT (-111.41085 44.53267)
268,867:ID:SNTL,867,ID,SNTL,Wildhorse Divide,ID,Bannock,170402080504,6480.0,42.75743,-112.47783,-8.0,WHDI1,NRCS,1979-10-01 00:00,2100-01-01 00:00,"[170402060902, 170402060903, 170402060904, 170...",59480,POINT (-112.47783 42.75743)


In [20]:
# Plot the study area and all the stations
sntl_plot = (
    # Plot study area polygon
    mhw_gdf.hvplot.polygons(
        geo=True,
        tiles="EsriNatGeo",
        fill_color=None,  # Clear inside
        line_color="black",
        line_width=1.5,
        title="SNOTEL Stations in Montana, Idaho, and Wyoming",
        width=900,
        height=700,
    )
    # SNOTEL Stations
    * sntl_station_gdf.hvplot.points(
        geo=True, 
        color="blue", 
        alpha=1,
        hover_cols = ['stationTriplet', 'name', 'countyName', 'huc'],
        size=40, 
        legend=False
    )
)

# Show plot
sntl_plot

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]
   .Points.I   :Points   [Longitude,Latitude]   (stationTriplet,name,countyName,huc)

The plot above shows the SNOTEL sites in MT (and one in WY that will be included). You can explore the map and see the names of stations, and get a sense of which ones fit into the study area or are very close.

### 3b: Filter stations

In [21]:
# use an inner join with the mhw_buff_clip_gdf

mhw_sntl_md_gdf = gpd.sjoin(
    sntl_station_gdf,
    mhw_buff_clip_gdf,
    how="inner",
    predicate="within"
).drop(columns="index_right")

In [22]:
# Inspect
print(f'There are {len(mhw_sntl_md_gdf)} SNOTEL stations within the study area')
mhw_sntl_md_gdf

There are 28 SNOTEL stations within the study area


,stationTriplet,stationId,stateCode,networkCode,name,dcoCode,countyName,huc,elevation,latitude,longitude,dataTimeZone,shefId,operator,beginDate,endDate,associatedHucs,pedonCode,geometry
0,916:MT:SNTL,916,MT,SNTL,Albro Lake,MT,Madison,100200050701,8500.0,45.59723,-111.95902,-8.0,ABRM8,NRCS,1996-09-01 00:00,2100-01-01 00:00,"[100200050601, 100200050702, 100200071101]",NaN,POINT (-111.95902 45.59723)
6,318:MT:SNTL,318,MT,SNTL,Beagle Springs,MT,Beaverhead,100200010501,8880.0,44.47147,-112.98191,-8.0,BEAM8,NRCS,1976-10-01 00:00,2100-01-01 00:00,"[100200010502, 100200010601, 170402160103, 170...",NaN,POINT (-112.98191 44.47147)
7,328:MT:SNTL,328,MT,SNTL,Beaver Creek,MT,Gallatin,100200070402,7820.0,44.94966,-111.35852,-8.0,BEVM8,NRCS,1966-10-01 00:00,2100-01-01 00:00,"[100200070401, 100200070403, 100200080105, 100...",NaN,POINT (-111.35852 44.94966)
9,347:MT:SNTL,347,MT,SNTL,Black Bear,MT,Gallatin,100200070202,8160.0,44.50832,-111.12803,-8.0,BLBM8,NRCS,1967-10-01 00:00,2100-01-01 00:00,"[170402020105, 170402020302, 170402020303, 170...",40626,POINT (-111.12803 44.50832)
12,355:MT:SNTL,355,MT,SNTL,Bloody Dick,MT,Beaverhead,100200011002,7570.0,45.16507,-113.50099,-8.0,BLOM8,NRCS,1976-10-01 00:00,2100-01-01 00:00,"[100200011001, 100200040301, 100200040302, 170...",NaN,POINT (-113.50099 45.16507)
15,365:MT:SNTL,365,MT,SNTL,Brackett Creek,MT,Gallatin,100700030403,7370.0,45.89107,-110.93851,-8.0,BRCM8,NRCS,1992-08-01 00:00,2100-01-01 00:00,"[100200080801, 100200081101, 100200081102, 100...",NaN,POINT (-110.93851 45.89107)
17,381:MT:SNTL,381,MT,SNTL,Calvert Creek,MT,Deer Lodge,100200040801,6410.0,45.88380,-113.32553,-8.0,CLVM8,NRCS,1974-10-01 00:00,2100-01-01 00:00,"[100200040607, 100200040608, 100200040803]",NaN,POINT (-113.32553 45.8838)
18,385:MT:SNTL,385,MT,SNTL,Carrot Basin,MT,Gallatin,100200080105,9200.0,44.96192,-111.29403,-8.0,CRRM8,NRCS,1966-10-01 00:00,2100-01-01 00:00,"[100200070401, 100200070402, 100200080106, 100...",NaN,POINT (-111.29403 44.96192)
20,403:MT:SNTL,403,MT,SNTL,Clover Meadow,MT,Madison,100200070803,8770.0,45.01788,-111.84560,-8.0,CMDM8,NRCS,1971-10-01 00:00,2100-01-01 00:00,"[100200030105, 100200030106, 100200030301, 100...",39520,POINT (-111.8456 45.01788)
28,436:MT:SNTL,436,MT,SNTL,Darkhorse Lake,MT,Beaverhead,100200040301,8930.0,45.17368,-113.58460,-8.0,DHLM8,NRCS,1977-08-01 00:00,2100-01-01 00:00,"[100200011002, 100200040303, 170602040802, 170...",NaN,POINT (-113.5846 45.17368)


In [ ]:
# Visualize
mhw_sntl_plot = mhw_sntl_md_gdf.hvplot(
    geo=True, 
    color="blue", 
    alpha=1,
    hover_cols = ['stationTriplet', 'name', 'countyName', 'huc'],
    size=40, 
    legend=False
)

# Overlay
train_sntl_plot = (
    mhw_buff_clip_plot
    *
    mhw_sntl_plot
).opts(title='SNOTEL Sites included in Training Dataset')

# Show
train_sntl_plot

:Overlay
   .WMTS.I      :WMTS   [Longitude,Latitude]
   .Polygons.I  :Polygons   [Longitude,Latitude]
   .WMTS.II     :WMTS   [Longitude,Latitude]
   .Polygons.II :Polygons   [Longitude,Latitude]
   .Points.I    :Points   [Longitude,Latitude]   (stationTriplet,name,countyName,huc)

Looks like there are 28 sites, including those that are just outside the boundary. By including those we picked up 4 additional sites, which is about 17% more datasets.

#### 3c: Test download func for one site

In [ ]:
# test the func
test_sites = snotel.get_snotel_data(
    '916:MT:SNTL', variables, 'daily', 
    begin_date='10/01/1990', end_date='06/01/2020')
test_sites

# it works! note: can query earlier start date than datasets run!

[{'stationTriplet': '916:MT:SNTL',
  'data': [{'stationElement': {'elementCode': 'PREC',
     'ordinal': 1,
     'durationName': 'DAILY',
     'dataPrecision': 2,
     'storedUnitCode': 'in',
     'originalUnitCode': 'in',
     'beginDate': '1996-09-11 00:00',
     'endDate': '2100-01-01 00:00',
     'derivedData': False},
    'values': [{'date': '1996-10-01', 'value': 0.0},
     {'date': '1996-10-02', 'value': 0.0},
     {'date': '1996-10-03', 'value': 0.0},
     {'date': '1996-10-04', 'value': 0.0},
     {'date': '1996-10-05', 'value': 0.0},
     {'date': '1996-10-06', 'value': 0.0},
     {'date': '1996-10-07', 'value': 0.0},
     {'date': '1996-10-08', 'value': 0.0},
     {'date': '1996-10-09', 'value': 0.0},
     {'date': '1996-10-10', 'value': 0.0},
     {'date': '1996-10-11', 'value': 0.0},
     {'date': '1996-10-12', 'value': 0.0},
     {'date': '1996-10-13', 'value': 0.1},
     {'date': '1996-10-14', 'value': 0.2},
     {'date': '1996-10-15', 'value': 0.3},
     {'date': '1996-

In [ ]:
# Test the parser

test_ds = snotel.parse_snotel_to_xarray(test_sites)
test_ds

<xarray.Dataset> Size: 347kB
Dimensions:  (station: 1, time: 8664)
Coordinates:
  * station  (station) object 8B '916:MT:SNTL'
  * time     (time) datetime64[us] 69kB 1996-09-11 1996-09-12 ... 2020-06-01
Data variables:
    PREC     (station, time) float64 69kB nan nan nan nan ... 23.0 23.0 23.3
    TMAX     (station, time) float64 69kB 60.1 63.7 63.0 51.8 ... 70.2 64.6 63.5
    TMIN     (station, time) float64 69kB 46.8 40.3 37.4 32.9 ... 43.2 40.3 38.3
    WTEQ     (station, time) float64 69kB nan nan nan nan ... 4.9 3.4 0.7 0.0

#### 3d: Download all sites

In [ ]:
# Set inputs to wrapper

# list of stations
station_list = mhw_sntl_md_gdf['stationTriplet'].to_list()

begin_date = '10/01/1990'
end_date='06/01/2020'

In [ ]:
# Test the download wrapper
test_triplets = station_list[:2]

test_ds = snotel.build_snotel_dataset(test_triplets, mhw_sntl_md_gdf, variables, 'daily', begin_date='10/01/1990', end_date='06/01/2020')
test_ds

Fetching data for 2 stations in one API call...


Merging datasets...


C:\Users\Raini\AppData\Local\Temp\ipykernel_25108\4147976594.py:78: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  sntl_stations_dataset = xr.concat(station_datasets, dim='station')


<xarray.Dataset> Size: 780kB
Dimensions:       (station: 2, time: 10837)
Coordinates:
  * station       (station) object 16B '916:MT:SNTL' '318:MT:SNTL'
    latitude      (station) float64 16B 45.6 44.47
    longitude     (station) float64 16B -112.0 -113.0
    elevation     (station) float64 16B 8.5e+03 8.88e+03
    station_name  (station) <U14 112B 'Albro Lake' 'Beagle Springs'
  * time          (time) datetime64[us] 87kB 1990-10-01 ... 2020-06-01
Data variables:
    PREC          (station, time) float64 173kB nan nan nan ... 12.7 12.7 12.8
    TMAX          (station, time) float64 173kB nan nan nan ... 73.0 66.9 65.3
    TMIN          (station, time) float64 173kB nan nan nan ... 44.4 44.2 40.5
    WTEQ          (station, time) float64 173kB nan nan nan nan ... 0.0 0.0 0.0

In [ ]:
# Download SNOTEL data for entire study area

mhw_sntl_ds = snotel.build_snotel_dataset(
    station_list, mhw_sntl_md_gdf, variables,
    'daily', begin_date='1989-10-01', end_date='2020-06-01')

Fetching data for 28 stations in one API call...


Merging datasets...


C:\Users\Raini\AppData\Local\Temp\ipykernel_25108\4147976594.py:78: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  sntl_stations_dataset = xr.concat(station_datasets, dim='station')


In [ ]:
# Check that stations were downloaded
print(len(station_list))
print(len(mhw_sntl_ds.station))

print(station_list)
mhw_sntl_ds.station

# One station is missing. That's 1286:MT:SNTL. A quick check on the API site showed that site began recording in September 2020, so it makes sense it wasn't downloaded.

28
27
['916:MT:SNTL', '318:MT:SNTL', '328:MT:SNTL', '347:MT:SNTL', '355:MT:SNTL', '365:MT:SNTL', '381:MT:SNTL', '385:MT:SNTL', '403:MT:SNTL', '436:MT:SNTL', '448:MT:SNTL', '487:MT:SNTL', '1287:MT:SNTL', '568:MT:SNTL', '578:MT:SNTL', '590:MT:SNTL', '603:MT:SNTL', '609:MT:SNTL', '656:MT:SNTL', '722:MT:SNTL', '929:MT:SNTL', '753:MT:SNTL', '754:MT:SNTL', '1286:MT:SNTL', '813:MT:SNTL', '924:MT:SNTL', '858:MT:SNTL', '384:WY:SNTL']


<xarray.DataArray 'station' (station: 27)> Size: 216B
array(['916:MT:SNTL', '318:MT:SNTL', '328:MT:SNTL', '347:MT:SNTL',
       '355:MT:SNTL', '365:MT:SNTL', '381:MT:SNTL', '385:MT:SNTL',
       '403:MT:SNTL', '436:MT:SNTL', '448:MT:SNTL', '487:MT:SNTL',
       '1287:MT:SNTL', '568:MT:SNTL', '578:MT:SNTL', '590:MT:SNTL',
       '603:MT:SNTL', '609:MT:SNTL', '656:MT:SNTL', '722:MT:SNTL',
       '929:MT:SNTL', '753:MT:SNTL', '754:MT:SNTL', '813:MT:SNTL',
       '924:MT:SNTL', '858:MT:SNTL', '384:WY:SNTL'], dtype=object)
Coordinates:
  * station       (station) object 216B '916:MT:SNTL' ... '384:WY:SNTL'
    latitude      (station) float64 216B 45.6 44.47 44.95 ... 44.66 44.61 44.72
    longitude     (station) float64 216B -112.0 -113.0 -111.4 ... -111.1 -110.5
    elevation     (station) float64 216B 8.5e+03 8.88e+03 ... 6.79e+03 7.87e+03
    station_name  (station) <U16 2kB 'Albro Lake' 'Beagle Springs' ... 'Canyon'

#### 3e: Save SNOTEL data

In [46]:
# save as NetCDF

# Run some compression
encoding = {var: dict(zlib=True, complevel=5) for var in mhw_sntl_ds.data_vars}

# save the file
mhw_sntl_ds.to_netcdf(os.path.join(raw_data_dir, 'mhw_snotel_1990-2020_RAW.nc'), encoding=encoding)


## Step 4: Match SNOTEL Sites to Bias-Corrected Quality-Controlled SNOTEL dataset

Numerous studies have highlighted issues with the SNOTEL datasets. Some issues include temperature biases due to an erroneous conversion factor used when converting sensor voltage to &degC, and physically impossible SWE values due to how snow can bridge the sensor and then fall on it while melting.

Various studies have developed fixes for these issues, and luckily, the bias corrected and quality controlled dataset (hereafter referred to as the BCQC SNOTEL dataset) developed by [Sun et al. 2019](https://agupubs.onlinelibrary.wiley.com/doi/abs/10.1029/2018JD030140) and [Yan et al. 2018](https://doi.org/10.1002/2017WR021290) is publicly available for use.

This step will match the BCQC SNOTEL records to the stations previously selected, and save that data as ready for further use.

The BCQC data can be downloaded at: [https://www.pnnl.gov/projects/distributed-hydrology-soil-vegetation-model/data-products]

#### Step 4a: Get BCQC data for the study area/time period

In [23]:
# set BCQC dir
bcqc_dir = Path(raw_data_dir, 'snotel', 'bcqc_data_v2', 'bcqc_data')
bcqc_dir

WindowsPath('C:/Users/raini/Documents/Graduate_School/EDA_Certificate/Summer/snow-drought-modeling/data/raw/snotel/bcqc_data_v2/bcqc_data')

In [32]:
# import BCQC data for applicable stations

bcqc_sntl_full_ds, excluded_list = snotel.import_bcqc_data(bcqc_dir, mhw_sntl_md_gdf)

# Check excluded stations and shape of loaded BCQC sites
print(f'Stations {excluded_list} were not found in the BCQC dataset')
bcqc_sntl_full_ds

Albro Lake found in BCQC dataset
Beagle Springs found in BCQC dataset
Beaver Creek found in BCQC dataset
Black Bear found in BCQC dataset
Bloody Dick found in BCQC dataset
Brackett Creek found in BCQC dataset
Calvert Creek found in BCQC dataset
Carrot Basin found in BCQC dataset
Clover Meadow found in BCQC dataset
Darkhorse Lake found in BCQC dataset
Divide found in BCQC dataset
Frohner Meadow found in BCQC dataset
JL Meadow not found in BCQC dataset
Lakeview Ridge found in BCQC dataset
Lick Creek found in BCQC dataset
Lone Mountain found in BCQC dataset
Lower Twin found in BCQC dataset
Madison Plateau found in BCQC dataset
Mule Creek found in BCQC dataset
Rocker Peak found in BCQC dataset
Sacajawea found in BCQC dataset
Short Creek found in BCQC dataset
Shower Falls found in BCQC dataset
Slagamelt Lakes not found in BCQC dataset
Tepee Creek found in BCQC dataset
West Yellowstone found in BCQC dataset
Whiskey Creek found in BCQC dataset
Canyon found in BCQC dataset


<string>:41: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'date' ('date',) The recommendation is to set join explicitly for this case.


Stations ['1287:MT:SNTL', '1286:MT:SNTL'] were not found in the BCQC dataset


<xarray.Dataset> Size: 20MB
Dimensions:          (stationTriplet: 26, date: 18628)
Coordinates:
  * date             (date) datetime64[ns] 149kB 1970-10-01 ... 2021-09-30
  * stationTriplet   (stationTriplet) object 208B '916:MT:SNTL' ... '384:WY:S...
    stationId        (stationTriplet) <U3 312B '916' '318' '328' ... '858' '384'
    name             (stationTriplet) <U16 2kB 'Albro Lake' ... 'Canyon'
    latitude         (stationTriplet) float64 208B 45.6 44.47 ... 44.61 44.72
    longitude        (stationTriplet) float64 208B -112.0 -113.0 ... -110.5
    beginDate        (stationTriplet) <U16 2kB '1996-09-01 00:00' ... '1960-1...
    endDate          (stationTriplet) <U16 2kB '2100-01-01 00:00' ... '2100-0...
Data variables:
    daily_precip_in  (stationTriplet, date) float64 4MB nan nan nan ... 0.0 0.0
    tmax_f           (stationTriplet, date) float64 4MB nan nan ... 50.98 56.13
    tmin_f           (stationTriplet, date) float64 4MB nan nan nan ... nan nan
    tavg_f           (stationTriplet, date) float64 4MB nan nan ... 32.44 25.23
    SWE              (stationTriplet, date) float64 4MB nan nan nan ... 0.0 0.0

26 of the 28 stations in the study area were found in the BCQC dataset. It's missing 1287:MT:SNTL and 1286:MT:SNTL. A quick check in the metadata shows that these two stations began recording data in 2017, which is quite close to the end of the BCQC dataset, so it makes sense that these stations are excluded.

In [33]:
# subset BCQC dataset to the 1990-2020 Water Years

bcqc_sntl_ds = bcqc_sntl_full_ds.sel(date = slice('1990-10-01', '2020-06-01'))

# Check that this worked
print("Start Date:", bcqc_sntl_ds.date.min().values)
print("End Date:  ", bcqc_sntl_ds.date.max().values)

Start Date: 1990-10-01T00:00:00.000000000
End Date:   2020-06-01T00:00:00.000000000


In [34]:
bcqc_sntl_ds

<xarray.Dataset> Size: 11MB
Dimensions:          (stationTriplet: 26, date: 10837)
Coordinates:
  * date             (date) datetime64[ns] 87kB 1990-10-01 ... 2020-06-01
  * stationTriplet   (stationTriplet) object 208B '916:MT:SNTL' ... '384:WY:S...
    stationId        (stationTriplet) <U3 312B '916' '318' '328' ... '858' '384'
    name             (stationTriplet) <U16 2kB 'Albro Lake' ... 'Canyon'
    latitude         (stationTriplet) float64 208B 45.6 44.47 ... 44.61 44.72
    longitude        (stationTriplet) float64 208B -112.0 -113.0 ... -110.5
    beginDate        (stationTriplet) <U16 2kB '1996-09-01 00:00' ... '1960-1...
    endDate          (stationTriplet) <U16 2kB '2100-01-01 00:00' ... '2100-0...
Data variables:
    daily_precip_in  (stationTriplet, date) float64 2MB nan nan nan ... 0.0 0.0
    tmax_f           (stationTriplet, date) float64 2MB nan nan ... 66.43 68.49
    tmin_f           (stationTriplet, date) float64 2MB nan nan ... 38.62 35.53
    tavg_f           (stationTriplet, date) float64 2MB nan nan ... 55.1 53.04
    SWE              (stationTriplet, date) float64 2MB nan nan nan ... 0.0 0.0

#### Step 4b: Save BCQC data for later use

In [35]:
# save as NetCDF

# change stationTriplets to str
bcqc_sntl_ds['stationTriplet'] = bcqc_sntl_ds['stationTriplet'].astype(str)

# save the file
bcqc_sntl_ds.to_netcdf(Path(cleaned_data_dir, 'bcqc_snotel_1990-2020_RAW.nc'))
